In [1]:
import torch
import gc

# 1. 파이썬의 쓰레기 수집기 가동
gc.collect()

# 2. PyTorch가 잡고 있는 GPU 메모리 찌꺼기 비우기
torch.cuda.empty_cache()

In [2]:
# 1. 필수 라이브러리 설치
!pip install -q -U transformers accelerate bitsandbytes peft huggingface_hub

# 2. 허깅페이스 인증 (원본 Gemma 2 모델을 다운받기 위해 필수)
from huggingface_hub import login
from google.colab import userdata
# 코랩 좌측 🔑 아이콘에 'HF_TOKEN'을 저장해두었거나, 아래 주석을 풀고 직접 입력하세요.
# login(token="hf_여기에_본인_토큰_입력")
login(token=userdata.get('HF_TOKEN'))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# 3. 경로 설정 (직접 업로드한 폴더 경로로 변경 🌟)
base_model_id = "google/gemma-2-9b-it"
adapter_path = "/content/lora_adapter" # <--- 드래그 앤 드롭한 폴더 이름

print("모델 로딩 중... (코랩 [런타임 유형]이 T4 GPU인지 꼭 확인하세요!)")

# 4. 양자화 및 모델 로드
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# 5. 직접 업로드한 펫 용품 지식(LoRA) 불러와서 결합
model = PeftModel.from_pretrained(model, adapter_path)
print("✅ 어댑터 결합 완료! 테스트 준비 끝.")

# 6. 대화 생성 함수 (최신 규격 적용)
def generate_response(instruction, user_input):
    prompt = (
        f"<start_of_turn>user\n"
        f"{instruction}\n\n"
        f"질문: {user_input}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("model\n")[-1].strip()

# 7. 실전 테스트
system_instruction = "당신은 펫 용품 전문 상담 챗봇입니다. 사용자의 질문에 맞는 적절한 답변이나 카테고리(recommend, domain_qa 등)를 출력하세요."

test_questions = [
    "고양이 모래 중에서 먼지 안 날리는 거 있어?",
    "저알러지 사료 중에 연어 성분 없는 거 있어?",
    "지금 소화불량일때 어떤 사료가 좋아?"
]

print("\n--- 🐾 코랩에서 챗봇 테스트 시작 ---")
for q in test_questions:
    print(f"👤 유저: {q}")
    answer = generate_response(system_instruction, q)
    print(f"🤖 챗봇: {answer}")
    print("-" * 40)

모델 로딩 중... (코랩 [런타임 유형]이 T4 GPU인지 꼭 확인하세요!)


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

✅ 어댑터 결합 완료! 테스트 준비 끝.

--- 🐾 코랩에서 챗봇 테스트 시작 ---
👤 유저: 고양이 모래 중에서 먼지 안 날리는 거 있어?
🤖 챗봇: recommend: 네, 먼지가 적은 '먼지 방지 모래'나 '응고형 모래'를 사용하면 고양이가 코를 핥는 횟수를 줄여 폐 질환을 예방할 수 있습니다. 답변이 도움이 되셨을까요? 더 궁금한 점은 언제든 물어봐 주세요!
----------------------------------------
👤 유저: 저알러지 사료 중에 연어 성분 없는 거 있어?
🤖 챗봇: recommend: 네, 연어 알러지가 있는 고양이를 위한 옵션으로 닭고기나 오리 등의 육류를 사용한 저알러지 사료가 많습니다. 성분표의 '가수분해 단백질' 항목을 확인하세요.
----------------------------------------
👤 유저: 지금 소화불량일때 어떤 사료가 좋아?
🤖 챗봇: recommend: 소화 잘되는 사료를 추천드립니다. 소화 효소가 첨가된 사료나 단백질 위주 사료를 급여하여 장의 염증을 줄여주는 것이 좋습니다. 답변이 도움이 되셨을까요? 더 궁금한 점은 언제든 물어봐 주세요!
----------------------------------------


In [3]:
# 1. OpenAI 라이브러리만 가볍게 추가 설치
!pip install -q openai

import os
from openai import OpenAI
from google.colab import userdata

# 2. OpenAI API 키 설정 (코랩 좌측 🔑 아이콘에 'OPENAI_API_KEY'로 저장했다고 가정)
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    client = OpenAI()
    print("🌐 GPT-4o-mini API 연결 완료!")
except Exception as e:
    print("⚠️ API 키를 찾을 수 없습니다. 코랩 비밀번호 탭을 확인하거나 코드에 직접 입력하세요.")
    # os.environ["OPENAI_API_KEY"] = "sk-여기에-직접-입력하셔도-됩니다"
    # client = OpenAI()

# 3. GPT-4o-mini 응답 생성 함수
def get_openai_response(instruction, user_input):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": instruction},
                {"role": "user", "content": user_input}
            ],
            temperature=0.1
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"OpenAI API 에러 발생: {e}"

# # 4. ⚔️ 실전 비교 테스트 실행!
# # 💡 Domain QA 전용 시스템 프롬프트 (상품 추천 금지, 전문 지식 위주)
# system_instruction = (
#     "당신은 반려동물(강아지, 고양이)의 건강, 영양, 행동, 양육 상식에 대해 답변하는 펫 전문 지식 챗봇입니다. "
#     "사용자의 질문에 대해 수의학적 근거와 올바른 양육 지식을 바탕으로 친절하고 명확하게 답변해 주세요. "
#     "단, 특정 상품(사료, 간식, 용품 등)의 브랜드나 제품명을 직접 추천하지 말고, 해결을 위한 '성분'이나 '관리 방법' 위주로 설명하세요."
# )

# # 💡 순수 도메인 지식(QA) 테스트 질문 20가지
# test_questions = [
#     # 1. 응급/독성 물질 (안전성 테스트)
#     "강아지가 초콜릿을 엄지손톱만큼 먹었는데 당장 병원에 가야 해?",
#     "강아지한테 사과나 배 같은 과일 줄 때 씨앗은 왜 꼭 빼야 해?",
#     "고양이한테 사람이 먹는 우유(락토스) 줘도 괜찮아?",

#     # 2. 질병 및 증상 (의료/건강 지식)
#     "노령견이 갑자기 물을 엄청 많이 마시는데 의심해 볼 만한 질병이 있어?",
#     "고양이가 켁켁거리면서 털뭉치(헤어볼)를 토하는데 자연스러운 현상이야?",
#     "강아지가 발바닥을 계속 핥아서 빨갛게 부었는데 피부병일까?",
#     "고양이가 갑자기 소변을 찔끔거리면서 화장실을 들락날락하는데 왜 그런 거야?",
#     "포메라니안이나 폼피츠 같은 견종이 슬개골 탈구가 잘 오는 이유가 뭐야?",

#     # 3. 행동 및 훈련 (심리/행동학)
#     "분리불안이 있는 강아지 혼자 두고 출근할 때 도움 되는 훈련법 있어?",
#     "고양이가 화장실 밖(이불 위)에 오줌을 싸는 이유가 뭘까?",
#     "고양이가 자꾸 밤마다 우다다를 하면서 우는데 이유가 뭐야?",
#     "강아지가 자기 똥을 먹는 식분증이 있는데 어떻게 고칠 수 있어?",
#     "고양이가 꼬리를 탁탁 치면서 바닥을 내리치는 건 어떤 기분 상태야?",

#     # 4. 연령별/생애주기 관리 (영양학 및 일반 상식)
#     "새끼 고양이 사료(키튼)는 언제부터 성묘용(어덜트)으로 바꿔주는 게 좋아?",
#     "수컷 강아지 중성화 수술을 하면 어떤 장단점이 있어?",
#     "사료 급여량 계산할 때 실내견 활동량은 보통 어떻게 기준을 잡아?",

#     # 5. 일상 케어 및 위생 (양육 상식)
#     "강아지 양치질은 일주일에 몇 번 정도 해주는 게 가장 적당해?",
#     "강아지 목욕은 한 달에 몇 번이 제일 적당해? 너무 자주 하면 안 좋아?",
#     "실내에서만 생활하는 고양이도 심장사상충 약을 매달 발라줘야 해?",
#     "고양이 예방접종(종합백신)은 어릴 때 맞고 나서 매년 꼭 추가로 맞혀야 해?"
# ]

# print("="*60)
# print(" 🐾 펫 용품 챗봇 성능 비교: My Gemma 2 vs GPT-4o-mini")
# print("="*60)

# for q in test_questions:
#     print(f"\n👤 유저 질문: {q}\n")

#     # 🌟 이전 셀에서 이미 만들어둔 내 모델의 함수를 그대로 재활용합니다!
#     gemma_ans = generate_response(system_instruction, q)
#     print(f"🐶 [내 파인튜닝 모델]:\n{gemma_ans}\n")

#     # GPT-4o-mini 호출
#     gpt_ans = get_openai_response(system_instruction, q)
#     print(f"🤖 [GPT-4o-mini]:\n{gpt_ans}\n")
#     print("-" * 60)

🌐 GPT-4o-mini API 연결 완료!


In [6]:
import json
import random
import pandas as pd
from openai import OpenAI
from google.colab import userdata

# 1. 환경 설정 및 심판(GPT-4o) 초기화
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

def evaluate_with_gpt4o(question, ground_truth, model_a_ans, model_b_ans):
    """
    GPT-4o를 심판으로 사용하여 두 모델의 답변을 정답지와 비교 채점합니다.
    """
    prompt = f"""
    당신은 반려동물 상담 전문 챗봇의 성능을 평가하는 심판입니다.
    질문과 정답지(Ground Truth)를 참고하여, 두 모델의 답변을 [정확성, 말투, 지시이행] 기준으로 평가하세요.

    [질문]: {question}
    [정답지]: {ground_truth}

    [모델 A의 답변]: {model_a_ans}
    [모델 B의 답변]: {model_b_ans}

    [평가 가이드라인]
    1. 정확성: 정답지의 핵심 지식(성분, 주의사항 등)을 빠뜨리지 않고 전달했는가?
    2. 말투: 전문적이고 다정하며, 학습 데이터 특유의 톤앤매너를 잘 유지했는가?
    3. 지시이행: 특정 브랜드 언급 금지 등 제약 사항을 잘 지켰는가?

    [결과 형식]
    반드시 아래 JSON 형식으로만 답변하세요:
    {{
        "model_a_score": (0~10점),
        "model_b_score": (0~10점),
        "winner": ("Model A" 또는 "Model B" 또는 "Tie"),
        "reason": "승자를 선택한 구체적인 이유"
    }}
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": "You are a professional AI model evaluator."},
                  {"role": "user", "content": prompt}],
        response_format={ "type": "json_object" }
    )
    return json.loads(response.choices[0].message.content)

# 2. 데이터셋 로드 및 샘플링
dataset_path = "train_dataset.jsonl"
eval_samples = []

with open(dataset_path, 'r', encoding='utf-8') as f:
    all_data = [json.loads(line) for line in f]
    eval_samples = random.sample(all_data, 100) # 시간/비용 관계상 5개만 먼저 테스트 (숫자 변경 가능)

# 3. 자동 평가 루프 실행
results = []
print(f"🧐 총 {len(eval_samples)}개의 샘플로 평가를 시작합니다...\n")

for i, data in enumerate(eval_samples):
    instr = data['instruction']
    q = data['input']
    gt = data['output']

    print(f"[{i+1}/{len(eval_samples)}] 질문 처리 중: {q[:30]}...")

    # 내 모델(Gemma 2) 답변 생성
    my_model_ans = generate_response(instr, q)

    # GPT-4o-mini 답변 생성
    gpt_mini_ans = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": instr}, {"role": "user", "content": q}]
    ).choices[0].message.content

    # GPT-4o 심판 채점
    judgment = evaluate_with_gpt4o(q, gt, my_model_ans, gpt_mini_ans)

    results.append({
        "질문": q,
        "정답지(GT)": gt,
        "내 모델(Gemma2)": my_model_ans,
        "GPT-4o-mini": gpt_mini_ans,
        "내 모델 점수": judgment['model_a_score'],
        "GPT mini 점수": judgment['model_b_score'],
        "승자": judgment['winner'],
        "이유": judgment['reason']
    })

# 4. 결과 리포트 출력
df = pd.DataFrame(results)
print("\n" + "="*50)
print("🏆 최종 평가 리포트")
print("="*50)
print(f"내 모델 평균 점수: {df['내 모델 점수'].mean():.2f}")
print(f"GPT mini 평균 점수: {df['GPT mini 점수'].mean():.2f}")
print(f"내 모델 승률: {(df['승자'] == 'Model A').mean()*100:.1f}%")

# 엑셀/CSV로 저장하여 다운로드 가능
df.to_csv("evaluation_report.csv", index=False, encoding='utf-8-sig')
print("\n✅ 상세 결과가 'evaluation_report.csv'로 저장되었습니다.")

# 결과 테이블 시각화 (코랩 전용)
display(df[['질문', '내 모델 점수', 'GPT mini 점수', '승자', '이유']])

🧐 총 100개의 샘플로 평가를 시작합니다...

[1/100] 질문 처리 중: 아키타 이누의 특징과 관리법을 알려줘....
[2/100] 질문 처리 중: **아기 고양이(자묘)**의 수염이 성묘보다 짧고 끝이...
[3/100] 질문 처리 중: 강아지가 산책 후 집에 돌아와서 '우다다' 하며 뛰어다...
[4/100] 질문 처리 중: 불리 쿠타의 특징과 관리법을 알려줘....
[5/100] 질문 처리 중: 강아지 식단에서 '비타민 D' 과잉 섭취 시 발생하는 ...
[6/100] 질문 처리 중: 급성 위확장-염전(GDV)' 시 발생하는 쇼크의 직접적...
[7/100] 질문 처리 중: 고양이 '흡수성 병변' 수술 후 사료 급여 시 주의할 ...
[8/100] 질문 처리 중: 고양이 '귀 세정' 후 귀 안쪽이 붉게 발적되었다면 의...
[9/100] 질문 처리 중: 고양이 '수염' 근처의 모낭염이 심해져 수염이 빠지기 ...
[10/100] 질문 처리 중: 고양이 '귀 소독' 후 귀 안의 세정액을 닦아낼 때 '...
[11/100] 질문 처리 중: 단두종 증후군' 환견이 더운 날씨에 유독 취약한 해부학...
[12/100] 질문 처리 중: 고양이가 가구를 스크래칭하는 행동을 멈추게 하려면?...
[13/100] 질문 처리 중: 고양이 '비대성 심근증(HCM)' 환묘가 개구호흡을 하...
[14/100] 질문 처리 중: 팔꿈치 이형성증'의 주요 원인인 '내측 구상돌기 분리(...
[15/100] 질문 처리 중: 생고기 위주의 식단(생식)을 할 때 가장 주의해야 할 ...
[16/100] 질문 처리 중: 면역 매개성 혈소판 감소증(IMTP)' 치료 시 '빈크...
[17/100] 질문 처리 중: 고양이가 사냥 놀이 도중 갑자기 자기 꼬리를 공격하는 ...
[18/100] 질문 처리 중: 고양이가 꼬리를 등 쪽으로 둥글게 말아 '물개'처럼 세...
[19/100] 질문 처리 중: 단쇄 지방산(SCFA)'이 강아지의 대장 건강에 기여하...
[20/100] 질문 처리

,질문,내 모델 점수,GPT mini 점수,승자,이유
0,아키타 이누의 특징과 관리법을 알려줘.,7,9,Model B,"Model A는 심장 판막 질환과 식단 관련 정보가 제공되었지만, 정답지에 언급된 ..."
1,**아기 고양이(자묘)**의 수염이 성묘보다 짧고 끝이 곱슬거리는 이유는?,6,9,Model B,Model B's answer is more accurate and covers t...
2,강아지가 산책 후 집에 돌아와서 '우다다' 하며 뛰어다니는 이유는?,9,8,Model A,Model A의 답변은 정답지의 내용을 거의 그대로 유지하며 정확성과 말투 측면에서...
3,불리 쿠타의 특징과 관리법을 알려줘.,8,9,Model B,"Model A는 불리 쿠타의 기원에 대한 정보에 오류가 있을 뿐 아니라, 정답지의 ..."
4,강아지 식단에서 '비타민 D' 과잉 섭취 시 발생하는 '전이성 석회화'의 위험성은?,9,8,Model A,"Model A는 정답지와 완전히 일치하는 정확한 정보를 제공하며, 매우 다정한 말투..."
...,...,...,...,...,...
95,"분리 불안' 환견이 보호자 외출 직전이 아닌, 귀가 직후에 더 격렬하게 반응하는 이유는?",8,9,Model B,"Model A는 반려견의 귀가 후 흥분을 '회사적 흥분'이라 잘못 설명했으며, 스트..."
96,"강아지 사료의 '무곡물(Grain-Free)' 식단, 정말 더 좋은가요?",8,9,Model B,"모델 A와 B 모두 정확성, 말투, 지시이행 면에서 좋은 성과를 보였지만, 모델 B..."
97,강아지도 꿈을 꾸나요?,9,8,Model A,"모델 A는 정답지와 거의 동일한 정보를 제공하고 있으며, 강아지의 수면 중 행동이 ..."
98,고양이 '허피스' 증상 완화를 위해 'L-라이신' 대신 권장되는 최신 지견?,7,8,Model B,모델 A는 면역력 강화에 도움이 되는 실제 권장 성분인 '베타글루칸'과 '락토페린'...
